# Lab 7.5 &mdash; Route: Decide What Happens Next

**Level:** Beginner &nbsp;|&nbsp; **Est. time:** 20 min &nbsp;|&nbsp; **Day 4 &middot; Module 7 &mdash; Task Automation with AI Agents**

### What you'll do
- Map an intent to a team from a fixed set (with an escape hatch)
- Escalate negative or unknown cases to a human
- Emit a label that drives the next branch of the workflow

> **How this lab works (near-real):** you have a `GROQ_API_KEY` in the repo `.env`. Read the **Concept**, fill the real `___` blanks in **Build it** (real pipeline logic, real tool bodies, the real draft/`create_agent` call), then **Run it for real** &mdash; a real Groq model drives the step over real tools &mdash; and **read the output/trace**. Finish with an open **Your turn**. There is **no auto-grader**; the goal is a working email agent and a trace you can read.

> **Framework note:** these labs use the **real** LangChain 1.x (`langchain`, `langchain-core`, `langchain-groq`) and a **real hosted model** (`ChatGroq("openai/gpt-oss-20b")` &mdash; reliable tool-calling via `create_agent`). If `GROQ_API_KEY` is unset, the run cells print how to set it instead of crashing. A `@tool` must **catch its own errors and return a string** &mdash; a tool that *raises* can abort the whole agent run. You are building the **email-drafting agent** (the client's Lab 4.1): it **drafts but never sends** &mdash; `send_email` is withheld and a human approves.

**Reference:** [Module 7 slides &mdash; Route — decide what happens next](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 7 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, time, socket, pathlib
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY (+ other keys)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-07-05")
os.makedirs(WORK, exist_ok=True)

def groq_ready():
    """True if a GROQ_API_KEY is configured -- the 'Run it for real' cells self-skip if not."""
    return bool(os.environ.get("GROQ_API_KEY"))

from langchain_groq import ChatGroq
# Day-4 provider: a REAL hosted model with reliable tool-calling via create_agent.
MODEL = "openai/gpt-oss-20b"
llm = ChatGroq(model=MODEL, temperature=0) if groq_ready() else None

def with_backoff(fn, tries=4):
    """Run fn(); retry with backoff on Groq rate limits (HTTP 429). Other errors propagate."""
    last = None
    for attempt in range(tries):
        try:
            return fn()
        except Exception as e:
            last = e
            if "429" in str(e) or "rate limit" in str(e).lower() or "rate_limit" in str(e).lower():
                wait = 5 * (attempt + 1)
                print(f"(rate-limited -- retrying in {wait}s)")
                time.sleep(wait)
                continue
            raise
    raise last

def print_trace(result):
    """Print a REAL agent message trace: tool calls the model made, tool observations, final answer."""
    for m in result["messages"]:
        for tc in (getattr(m, "tool_calls", None) or []):
            print("TOOL CALL:", tc["name"], tc["args"])
        if type(m).__name__ == "ToolMessage":
            print("OBS:", str(m.content)[:200])
        elif str(getattr(m, "content", "")).strip():
            print(type(m).__name__, ":", str(m.content)[:300])

if groq_ready():
    print("Groq ready | model:", MODEL, "| WORK:", WORK)
else:
    print("GROQ_API_KEY not set -- add it to the repo .env (free key at console.groq.com).")
    print("(The 'Run it for real' cells will print this note instead of crashing.)  WORK:", WORK)

## Concept
**Route** decides what happens next and emits a **label from a fixed set** that drives a branch
(deck slide 11): which team, how urgent, auto-handle or escalate. The **closed list** is the trick
that makes an LLM a reliable classifier &mdash; include an **escape hatch** (`other` / `unsure`), and
a **fallback path** that routes low-confidence or high-stakes cases to a human. (Deterministic
routing here; routing to a specialist *agent* is Module 8.)

In [ ]:
TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}
print("closed label set -> team:", TEAMS)

## Build it
Complete `route`: pick the team from the closed map, and decide when to **escalate** to a human.

In [ ]:
TEAMS = {"refund": "billing", "delivery": "logistics", "cancel": "billing", "other": "general"}

def route(record):
    intent = record.get("intent", "other")
    team = ___   # TODO: the team for intent, defaulting to "general" (the escape hatch)
    # escalate to a human when the customer is unhappy OR the intent is unknown
    escalate = ___   # TODO: True if sentiment is negative or intent not in TEAMS
    return {"team": team, "escalate": escalate}

try:
    print("refund   ->", route({"intent": "refund", "sentiment": "neutral"}))
    print("delivery ->", route({"intent": "delivery", "sentiment": "negative"}))
    print("unknown  ->", route({"intent": "mystery", "sentiment": "neutral"}))
except Exception as e:
    print("(Fill the ___ blanks above, then re-run.)", type(e).__name__)

## What to notice
- An unknown intent falls back to `general` via the **escape hatch** &mdash; the router never crashes on a label it hasn't seen.
- A **negative** sentiment escalates to a human even when the team is known &mdash; the high-stakes fallback path.
- The emitted `{team, escalate}` is the label the rest of the workflow branches on; keep it small and closed.

## Your turn (open task &mdash; no grader)
Add a new intent/team pair (e.g. `"billing_query": "billing"`) or a stricter escalation rule (escalate
any `refund` over a threshold). **What good looks like:** known intents route deterministically, unknowns
fall to `general`, and the cases you'd want a human to see are the ones that get `escalate=True`.

---
### Key takeaway
Routing makes one agent the front door to a whole system. Routing to the right specialist AGENT is the bridge to Module 8 -- for now it's the label that drives the branch.

[&#8592; All Module 7 labs](./index.html) &nbsp;&middot;&nbsp; [Module 7 slides](../../presentation/day4-module7-task-automation.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>